# Redispatch ↔ plant_matcher join + event-cause analysis

Joins the geocoded plant list (`plant_matcher_output.xlsx`) onto the raw ÜNB redispatch event tables and explores the distribution of event causes (`GRUND_DER_MASSNAHME`), energy carriers, direction, and timing.

**Join key:** `BETROFFENE_ANLAGE` (redispatch) ↔ `plant_name` (matcher output).

**Caveats baked into the redispatch CSVs (handled below):**
- Encoding is mojibake'd — `Jänschwalde` is stored as `J�nschwalde` in BOTH files, so a direct string match works.
- `RICHTUNG` has two visually identical levels that differ by a trailing space → normalised.
- `BETROFFENE_ANLAGE` mixes real plants with markets (`Börse`), traders (`EnBW Trading GmbH`) and reserve labels (`(KapRes)`, `(NetzRes)`) — they survive the join only if `plant_matcher_output` has a row for them.
- The 2013–2020 export and the 2021–2025 export have identical schemas and are concatenated.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

DATA_DIR = Path('.').resolve()
MATCHER_FILE = DATA_DIR / 'plant_matcher_output.xlsx'
REDISPATCH_FILES = [
    DATA_DIR / '2025-09-17 Redispatch Export 2013-2020.csv',
    DATA_DIR / 'Redispatch_Daten(3).csv',
]
for p in [MATCHER_FILE, *REDISPATCH_FILES]:
    assert p.exists(), p
DATA_DIR

## 1. Load the matcher output

In [ ]:
matcher = pd.read_excel(MATCHER_FILE)
print('matcher rows:', len(matcher), '— columns:', matcher.shape[1])
matcher[['plant_name', 'primaerenergieart', 'final_lat', 'final_lon',
         'final_confidence', 'final_source', 'n_events', 'sources']].head()

In [ ]:
# Slim the matcher table to the columns we actually need on the join.
matcher_slim = matcher[[
    'plant_name', 'primaerenergieart', 'final_lat', 'final_lon',
    'final_confidence', 'final_source', 'n_events', 'sources'
]].copy()
matcher_slim = matcher_slim.rename(columns={
    'final_lat': 'lat', 'final_lon': 'lon',
    'final_source': 'geocode_source',
    'n_events': 'n_events_matcher',
    'primaerenergieart': 'primaerenergieart_matcher',
})

dup = matcher_slim['plant_name'].duplicated().sum()
print(f'duplicate plant_name in matcher: {dup}')
matcher_slim.head(3)

## 2. Load + clean the redispatch tables

Both CSVs are semicolon-separated, German date format, and share the mojibake encoding. We load them as bytes-passthrough (`utf-8`), strip whitespace, parse the datetimes, and combine.

In [ ]:
def load_redispatch(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path, sep=';', encoding='utf-8',
        on_bad_lines='skip', low_memory=False,
    )
    # Strip stray whitespace from object columns (the dataset has a phantom
    # trailing space in some RICHTUNG entries that splits one category in two).
    for c in df.select_dtypes(include='object').columns:
        df[c] = df[c].astype(str).str.strip().replace({'nan': np.nan})
    # Datetimes — German DD.MM.YYYY HH:MM
    df['ts_start'] = pd.to_datetime(
        df['BEGINN_DATUM'] + ' ' + df['BEGINN_UHRZEIT'].fillna('00:00'),
        format='%d.%m.%Y %H:%M', errors='coerce')
    df['ts_end'] = pd.to_datetime(
        df['ENDE_DATUM'] + ' ' + df['ENDE_UHRZEIT'].fillna('00:00'),
        format='%d.%m.%Y %H:%M', errors='coerce')
    df['duration_h'] = (df['ts_end'] - df['ts_start']).dt.total_seconds() / 3600
    for c in ['MITTLERE_LEISTUNG_MW', 'MAXIMALE_LEISTUNG_MW', 'GESAMTE_ARBEIT_MWH']:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    df['source_file'] = path.name
    return df

rd_parts = [load_redispatch(p) for p in REDISPATCH_FILES]
for p, df in zip(REDISPATCH_FILES, rd_parts):
    print(f'{p.name}: {len(df):>7,} rows, {df["ts_start"].min().date()} → {df["ts_start"].max().date()}')

rd = pd.concat(rd_parts, ignore_index=True)
print('\ncombined:', len(rd), 'rows')

In [ ]:
# Quick before/after on the RICHTUNG dedupe to show the bug is real.
raw = pd.concat(
    [pd.read_csv(p, sep=';', encoding='utf-8', usecols=['RICHTUNG'], low_memory=False)
     for p in REDISPATCH_FILES],
    ignore_index=True)
print('Raw RICHTUNG levels (note the duplicate caused by trailing space):')
print(raw['RICHTUNG'].value_counts(dropna=False))
print('\nAfter strip():')
print(rd['RICHTUNG'].value_counts(dropna=False))

## 3. Join on plant name

Left join from redispatch events → matcher table. Anything that fails to match is either (a) a non-plant entity like `Börse`, (b) a reserve label not in the matcher output, or (c) a name variant the matcher didn't capture.

In [ ]:
joined = rd.merge(
    matcher_slim,
    left_on='BETROFFENE_ANLAGE',
    right_on='plant_name',
    how='left',
    indicator=True,
)

n_total = len(joined)
n_matched = (joined['_merge'] == 'both').sum()
print(f'events matched: {n_matched:,} / {n_total:,} = {n_matched/n_total:.1%}')
print(f'distinct plant names in redispatch : {rd["BETROFFENE_ANLAGE"].nunique():,}')
print(f'    of which matched              : {joined.loc[joined["_merge"]=="both", "BETROFFENE_ANLAGE"].nunique():,}')
print(f'    of which unmatched            : {joined.loc[joined["_merge"]=="left_only", "BETROFFENE_ANLAGE"].nunique():,}')

In [ ]:
# Inspect the biggest unmatched plant names — these are the candidates that
# *should* be in the matcher output but slipped through.
unmatched = (
    joined.loc[joined['_merge'] == 'left_only']
          .groupby('BETROFFENE_ANLAGE')
          .size()
          .sort_values(ascending=False)
          .head(25)
)
unmatched.to_frame('n_events')

In [ ]:
# Persist the join for downstream notebooks.
OUT = DATA_DIR / 'redispatch_joined.parquet'
joined.drop(columns=['_merge']).to_parquet(OUT, index=False)
print('wrote', OUT, '—', OUT.stat().st_size // 1024, 'KB')

### 3a. High-confidence filtered CSV

Drop everything that isn't a confident plant match: keep only rows where `_merge == 'both'` and `final_confidence > 0.5`. Written as CSV (semicolon, UTF-8 with BOM so Excel opens the German columns correctly).

In [ ]:
CONF_THRESHOLD = 0.5

high_conf = joined[(joined['_merge'] == 'both') &
                   (joined['final_confidence'] > CONF_THRESHOLD)].copy()
high_conf = high_conf.drop(columns=['_merge'])

# Reason filter: keep *-bedingt Redispatch (Strom-, Spannungs-, Strom-und-Spannungs-),
# drop Countertrade + test runs (Probestart/Funktionstest/Testfahrt/Probeabruf/Probefahrt)
# + Unbekannt + Gezielter Leistungsausgleich
reason = high_conf['GRUND_DER_MASSNAHME'].astype(str)
reason_mask = (reason.str.contains('bedingt', case=False, na=False)
               & ~reason.str.contains('countertrade', case=False, na=False))
n_before_reason = len(high_conf)
high_conf = high_conf[reason_mask]

OUT_CSV = DATA_DIR / 'redispatch_joined_high_confidence.csv'
high_conf.to_csv(OUT_CSV, sep=';', index=False, encoding='utf-8-sig')

print(f'after confidence > {CONF_THRESHOLD}: {n_before_reason:>7,} events')
print(f'after reason filter             : {len(high_conf):>7,} events  '
      f'({len(high_conf)/len(joined):.1%} of all)')
print(f'reasons remaining: {sorted(high_conf["GRUND_DER_MASSNAHME"].unique())}')
print(f'distinct plants kept: {high_conf["BETROFFENE_ANLAGE"].nunique()}')
print(f'wrote {OUT_CSV.name} — {OUT_CSV.stat().st_size // 1024:,} KB')

## 4. Distribution of event causes (`GRUND_DER_MASSNAHME`)

In [ ]:
cause_counts = joined['GRUND_DER_MASSNAHME'].value_counts()
print(cause_counts)

fig, ax = plt.subplots(figsize=(9, 5))
cause_counts.sort_values().plot.barh(ax=ax, color='#3a6ea5')
ax.set_xlabel('Number of events')
ax.set_ylabel('')
ax.set_title('Redispatch events by cause (GRUND_DER_MASSNAHME), 2013–2025')
for p in ax.patches:
    ax.annotate(f'{int(p.get_width()):,}',
                (p.get_width(), p.get_y() + p.get_height()/2),
                ha='left', va='center', fontsize=8, xytext=(3, 0),
                textcoords='offset points')
plt.tight_layout()
plt.show()

### 4a. Cause × energy carrier

The cause distribution looks different for conventional vs renewable plants — that's the key analytical contrast for the seminar (curtailment-style events hit renewables, voltage/current-bedingt RD hits the big conventional fleet).

In [ ]:
# Use the matcher-side energy carrier when matched, else the redispatch one.
joined['carrier'] = joined['primaerenergieart_matcher'].fillna(joined['PRIMAERENERGIEART'])
ct = (pd.crosstab(joined['GRUND_DER_MASSNAHME'], joined['carrier'])
        .sort_values(by=list(joined['carrier'].dropna().unique()), ascending=False)
        .reindex(joined['GRUND_DER_MASSNAHME'].value_counts().index))
ct

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ct.iloc[::-1].plot.barh(ax=ax, stacked=True,
                         color=['#cc4b3a', '#3aa856', '#888'])
ax.set_xlabel('Number of events')
ax.set_ylabel('')
ax.set_title('Event cause × primary energy carrier')
ax.legend(title='', loc='lower right')
plt.tight_layout()
plt.show()

## 5. Direction (up- vs down-regulation)

In [ ]:
dir_ct = pd.crosstab(joined['GRUND_DER_MASSNAHME'], joined['RICHTUNG'])
dir_ct = dir_ct.loc[dir_ct.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(10, 5))
dir_ct.iloc[::-1].plot.barh(ax=ax, stacked=True, color=['#e09d3c', '#3a6ea5'])
ax.set_xlabel('Number of events')
ax.set_title('Up- vs down-regulation by cause')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Events over time

Monthly counts, stacked by cause. The 2021 transition (Redispatch 2.0 / NABEG) is visible as a structural break.

In [ ]:
ts = (joined.dropna(subset=['ts_start'])
             .assign(month=lambda d: d['ts_start'].dt.to_period('M').dt.to_timestamp())
             .groupby(['month', 'GRUND_DER_MASSNAHME']).size().unstack(fill_value=0))

# Keep top 6 causes by total volume, group the rest as 'Other'.
top = ts.sum().sort_values(ascending=False).head(6).index
ts_top = ts[top].copy()
ts_top['Other'] = ts.drop(columns=top).sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
ts_top.plot.area(ax=ax, linewidth=0, alpha=0.9)
ax.set_title('Monthly redispatch events by cause')
ax.set_ylabel('Events / month')
ax.set_xlabel('')
ax.axvline(pd.Timestamp('2021-10-01'), color='k', ls='--', lw=0.8)
ax.text(pd.Timestamp('2021-10-15'), ax.get_ylim()[1]*0.95,
        'Redispatch 2.0 (Oct 2021)', fontsize=8, va='top')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Energy volume by cause

Event counts overstate small voltage-RD interventions; `GESAMTE_ARBEIT_MWH` is the right yardstick for grid-stress magnitude.

In [ ]:
energy_by_cause = (joined.groupby('GRUND_DER_MASSNAHME')['GESAMTE_ARBEIT_MWH']
                          .sum().sort_values(ascending=True) / 1e3)  # GWh

fig, ax = plt.subplots(figsize=(9, 5))
energy_by_cause.plot.barh(ax=ax, color='#5a4d80')
ax.set_xlabel('Total redispatched energy (GWh)')
ax.set_title('Energy volume by cause, 2013–2025')
for p in ax.patches:
    ax.annotate(f'{p.get_width():,.0f}',
                (p.get_width(), p.get_y() + p.get_height()/2),
                ha='left', va='center', fontsize=8, xytext=(3, 0),
                textcoords='offset points')
plt.tight_layout()
plt.show()

## 8. Top plants — who actually gets dispatched

In [ ]:
top_plants = (joined[joined['_merge'] == 'both']
              .groupby(['BETROFFENE_ANLAGE', 'carrier']).size()
              .sort_values(ascending=False).head(25)
              .reset_index(name='n_events'))
top_plants

In [ ]:
colors = {'Konventionell': '#cc4b3a', 'Erneuerbar': '#3aa856', 'Sonstiges': '#888'}
fig, ax = plt.subplots(figsize=(10, 7))
tp = top_plants.iloc[::-1]
ax.barh(tp['BETROFFENE_ANLAGE'], tp['n_events'],
        color=[colors.get(c, '#888') for c in tp['carrier']])
ax.set_xlabel('Events 2013–2025')
ax.set_title('Top 25 plants by redispatch event count')
handles = [plt.Rectangle((0,0),1,1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), loc='lower right')
plt.tight_layout()
plt.show()

## 9. Geographic distribution of events

Each matched plant gets a marker sized by event count, coloured by carrier.

In [ ]:
geo = (joined[joined['_merge'] == 'both']
       .dropna(subset=['lat', 'lon'])
       .groupby(['BETROFFENE_ANLAGE', 'lat', 'lon', 'carrier'])
       .size().reset_index(name='n_events'))

fig, ax = plt.subplots(figsize=(8, 9))
for carrier, color in colors.items():
    g = geo[geo['carrier'] == carrier]
    ax.scatter(g['lon'], g['lat'],
               s=np.sqrt(g['n_events']) * 2.5,
               c=color, alpha=0.55, edgecolor='white', linewidth=0.3,
               label=f'{carrier} (n={len(g)})')
ax.set_xlim(5.5, 15.5)
ax.set_ylim(47.0, 55.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Matched redispatch plants — marker size ∝ √(event count)')
ax.legend(loc='lower left')
ax.set_aspect(1.5)
plt.tight_layout()
plt.show()

## 10. TSO (ANWEISENDER_UENB) view

In [ ]:
tso_ct = pd.crosstab(joined['ANWEISENDER_UENB'], joined['GRUND_DER_MASSNAHME'])
tso_ct = tso_ct.loc[tso_ct.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(11, 5))
tso_ct.plot.bar(ax=ax, stacked=True, width=0.8)
ax.set_ylabel('Events')
ax.set_xlabel('')
ax.set_title('Events by instructing TSO, stacked by cause')
ax.legend(loc='upper right', fontsize=7, ncol=2)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 11. Events by matcher confidence score

How many of the joined events are tied to high-vs-low confidence plant matches? Buckets the `final_confidence` field from the matcher. Unmatched events get a synthetic `unmatched` bucket so they don't disappear.

Reading guide: events above ≈0.7 are trustworthy locations; the 0–0.5 tail is where the geocoding is hand-wavy and you should be cautious about geographic claims.

In [ ]:
bins = [-0.001, 0.0, 0.2, 0.4, 0.6, 0.8, 0.95, 1.0]
labels = ['0', '(0, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 0.95]', '(0.95, 1.0]']

buckets = pd.cut(joined['final_confidence'], bins=bins, labels=labels)
buckets = buckets.cat.add_categories(['unmatched']).fillna('unmatched')
joined['conf_bucket'] = buckets

events_by_conf = (joined.groupby('conf_bucket', observed=True)
                        .agg(n_events=('GRUND_DER_MASSNAHME', 'size'),
                             energy_gwh=('GESAMTE_ARBEIT_MWH', lambda s: s.sum()/1e3),
                             n_plants=('BETROFFENE_ANLAGE', 'nunique'))
                        .reindex(['unmatched'] + labels))
events_by_conf['share_events'] = events_by_conf['n_events'] / events_by_conf['n_events'].sum()
events_by_conf

In [ ]:
bucket_colors = ['#999', '#d96459', '#e09d3c', '#f0c93b', '#bfd96b', '#6abf69', '#3a8c4d', '#1f5e2e']
ordered = events_by_conf.loc[['unmatched'] + labels]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(len(ordered)), ordered['n_events'], color=bucket_colors)
axes[0].set_xticks(range(len(ordered)))
axes[0].set_xticklabels(ordered.index, rotation=30, ha='right')
axes[0].set_ylabel('Number of events')
axes[0].set_title('Redispatch events by matcher confidence bucket')
for i, (n, s) in enumerate(zip(ordered['n_events'], ordered['share_events'])):
    axes[0].annotate(f'{int(n):,}\n{s:.0%}', (i, n), ha='center', va='bottom', fontsize=8)
axes[0].set_ylim(0, ordered['n_events'].max() * 1.18)

axes[1].bar(range(len(ordered)), ordered['energy_gwh'], color=bucket_colors)
axes[1].set_xticks(range(len(ordered)))
axes[1].set_xticklabels(ordered.index, rotation=30, ha='right')
axes[1].set_ylabel('Total energy (GWh)')
axes[1].set_title('Redispatched energy by confidence bucket')
for i, e in enumerate(ordered['energy_gwh']):
    axes[1].annotate(f'{e:,.0f}', (i, e), ha='center', va='bottom', fontsize=8)
axes[1].set_ylim(0, ordered['energy_gwh'].max() * 1.12)

plt.tight_layout()
plt.show()

In [ ]:
# Cause × confidence — does the low-confidence tail concentrate on any one cause?
cause_conf = (pd.crosstab(joined['GRUND_DER_MASSNAHME'], joined['conf_bucket'])
                .reindex(columns=['unmatched'] + labels, fill_value=0))
cause_conf = cause_conf.loc[cause_conf.sum(axis=1).sort_values(ascending=False).index]
cause_conf_pct = cause_conf.div(cause_conf.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 5))
cause_conf_pct.iloc[::-1].plot.barh(ax=ax, stacked=True, color=bucket_colors, width=0.85)
ax.set_xlabel('Share of events')
ax.set_title('Confidence-bucket mix within each event cause')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8, title='confidence')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

## 12. Distribution of unmatched events

~16 % of events don't match a plant. Classify them so we can see whether they're truly missing geocoding (and should be added to the matcher), or whether they are genuinely non-plant entries that *can't* be located on a map.

Categories:
- **market** — `Börse`, traders, balancing groups (no physical location)
- **multi-plant bundle** — comma-separated lists of plants (legacy aggregated reporting)
- **reserve label** — `(KapRes)`, `(NetzRes)`, `(bnBm)` suffixes
- **other** — single-plant names the matcher missed (these are the actionable ones)

In [ ]:
unmatched_ev = joined.loc[joined['_merge'] == 'left_only'].copy()

def classify_unmatched(name: str) -> str:
    if not isinstance(name, str):
        return 'other'
    low = name.lower()
    if low.startswith('börse') or low.startswith('b�rse') or 'trading' in low \
       or 'bilanzkreis' in low or 'handel' in low:
        return 'market'
    if ',' in name:
        return 'multi-plant bundle'
    if any(tag in name for tag in ['(KapRes)', '(NetzRes)', '(bnBm)']):
        return 'reserve label'
    return 'other'

unmatched_ev['umatch_class'] = unmatched_ev['BETROFFENE_ANLAGE'].map(classify_unmatched)

unmatched_dist = (unmatched_ev.groupby('umatch_class')
                              .agg(n_events=('GRUND_DER_MASSNAHME', 'size'),
                                   energy_gwh=('GESAMTE_ARBEIT_MWH', lambda s: s.sum()/1e3),
                                   n_distinct_names=('BETROFFENE_ANLAGE', 'nunique'))
                              .sort_values('n_events', ascending=False))
unmatched_dist['share_of_unmatched'] = unmatched_dist['n_events'] / unmatched_dist['n_events'].sum()
unmatched_dist['share_of_all_events'] = unmatched_dist['n_events'] / len(joined)
unmatched_dist

In [ ]:
cls_colors = {'market': '#8a6dc0', 'multi-plant bundle': '#cc4b3a',
              'reserve label': '#e09d3c', 'other': '#3a6ea5'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ud = unmatched_dist.copy()
axes[0].bar(ud.index, ud['n_events'], color=[cls_colors[c] for c in ud.index])
axes[0].set_ylabel('Number of unmatched events')
axes[0].set_title(f"Unmatched events by category  (total {len(unmatched_ev):,})")
for i, (n, s) in enumerate(zip(ud['n_events'], ud['share_of_unmatched'])):
    axes[0].annotate(f'{int(n):,}\n{s:.0%}', (i, n), ha='center', va='bottom', fontsize=8)
axes[0].set_ylim(0, ud['n_events'].max() * 1.18)
plt.setp(axes[0].get_xticklabels(), rotation=20, ha='right')

axes[1].bar(ud.index, ud['energy_gwh'], color=[cls_colors[c] for c in ud.index])
axes[1].set_ylabel('Total energy (GWh)')
axes[1].set_title('Unmatched energy by category')
for i, e in enumerate(ud['energy_gwh']):
    axes[1].annotate(f'{e:,.0f}', (i, e), ha='center', va='bottom', fontsize=8)
axes[1].set_ylim(0, ud['energy_gwh'].max() * 1.15)
plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# What causes drive the unmatched stream? And what carrier (per the redispatch
# side, since the matcher side is null for these)?
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cause_unmatched = unmatched_ev['GRUND_DER_MASSNAHME'].value_counts()
cause_unmatched.sort_values().plot.barh(ax=axes[0], color='#3a6ea5')
axes[0].set_xlabel('Unmatched events')
axes[0].set_title('Unmatched events by cause')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_width()):,}',
                     (p.get_width(), p.get_y() + p.get_height()/2),
                     ha='left', va='center', fontsize=8, xytext=(3, 0),
                     textcoords='offset points')

carrier_unmatched = unmatched_ev['PRIMAERENERGIEART'].fillna('—').value_counts()
carrier_colors = {'Konventionell': '#cc4b3a', 'Erneuerbar': '#3aa856',
                  'Sonstiges': '#888', '—': '#ccc'}
axes[1].bar(carrier_unmatched.index, carrier_unmatched.values,
            color=[carrier_colors.get(c, '#888') for c in carrier_unmatched.index])
axes[1].set_ylabel('Unmatched events')
axes[1].set_title('Unmatched events by PRIMAERENERGIEART (redispatch-side)')
for i, v in enumerate(carrier_unmatched.values):
    axes[1].annotate(f'{int(v):,}', (i, v), ha='center', va='bottom', fontsize=8)
axes[1].set_ylim(0, carrier_unmatched.max() * 1.12)

plt.tight_layout()
plt.show()

In [ ]:
# Top 25 unmatched names AND their classification — these are the data-quality
# leads. Rows in the "other" class are the matcher's true misses; everything
# else is unmatchable by design.
top_unmatched = (unmatched_ev.groupby(['BETROFFENE_ANLAGE', 'umatch_class'])
                              .agg(n_events=('GRUND_DER_MASSNAHME', 'size'),
                                   energy_gwh=('GESAMTE_ARBEIT_MWH', lambda s: s.sum()/1e3))
                              .sort_values('n_events', ascending=False)
                              .head(25)
                              .reset_index())

fig, ax = plt.subplots(figsize=(11, 8))
tp = top_unmatched.iloc[::-1]
ax.barh(tp['BETROFFENE_ANLAGE'], tp['n_events'],
        color=[cls_colors[c] for c in tp['umatch_class']])
ax.set_xlabel('Events')
ax.set_title('Top 25 unmatched names, coloured by category')
handles = [plt.Rectangle((0,0),1,1, color=c) for c in cls_colors.values()]
ax.legend(handles, cls_colors.keys(), loc='lower right')
plt.tight_layout()
plt.show()

top_unmatched

## 13. Sanity summary

In [ ]:
summary = pd.Series({
    'events total': len(joined),
    'events matched to a plant': int((joined['_merge'] == 'both').sum()),
    'match rate (events)': f"{(joined['_merge'] == 'both').mean():.1%}",
    'distinct plants in redispatch': rd['BETROFFENE_ANLAGE'].nunique(),
    'distinct plants matched': joined.loc[joined['_merge']=='both', 'BETROFFENE_ANLAGE'].nunique(),
    'plants in matcher output': len(matcher_slim),
    'plants in matcher w/ coords': int(matcher_slim['lat'].notna().sum()),
    'date range': f"{joined['ts_start'].min().date()} → {joined['ts_start'].max().date()}",
    'total redispatched energy (TWh)': f"{joined['GESAMTE_ARBEIT_MWH'].sum()/1e6:.2f}",
})
summary.to_frame('value')

## 14. Matcher quality — `final_confidence` & `final_source`

Breakdown of the matched redispatch events by the matcher's geocoding **confidence** and the **source** it pulled the coordinates from. For each, two bar charts: the number of distinct **plants** in the bucket and the number of **events** tied to them.

Computed on matched rows only (`_merge == 'both'`), so every row carries a confidence and a source. Note: the matcher's `final_source` is carried into the join under the name `geocode_source` (renamed in §1); `final_confidence` keeps its name.

In [ ]:
matched = joined[joined['_merge'] == 'both'].copy()

conf_bins = [-0.001, 0.0, 0.2, 0.4, 0.6, 0.8, 0.95, 1.0]
conf_labels = ['0', '(0, 0.2]', '(0.2, 0.4]', '(0.4, 0.6]', '(0.6, 0.8]', '(0.8, 0.95]', '(0.95, 1.0]']
matched['conf_bucket'] = pd.cut(matched['final_confidence'], bins=conf_bins, labels=conf_labels)

conf_summary = (matched.groupby('conf_bucket', observed=False)
                       .agg(n_plants=('BETROFFENE_ANLAGE', 'nunique'),
                            n_events=('GRUND_DER_MASSNAHME', 'size'))
                       .reindex(conf_labels))
conf_summary['share_events'] = conf_summary['n_events'] / conf_summary['n_events'].sum()
conf_summary['share_plants'] = conf_summary['n_plants'] / conf_summary['n_plants'].sum()
conf_summary

In [ ]:
conf_colors = ['#d96459', '#e09d3c', '#f0c93b', '#bfd96b', '#6abf69', '#3a8c4d', '#1f5e2e']
x = range(len(conf_summary))

# --- Plants by confidence bucket ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, conf_summary['n_plants'], color=conf_colors)
ax.set_title('Distinct matched plants by confidence bucket')
ax.set_ylabel('Number of plants')
for i, v in enumerate(conf_summary['n_plants']):
    ax.annotate(f'{int(v):,}', (i, v), ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, conf_summary['n_plants'].max() * 1.15)
ax.set_xticks(list(x))
ax.set_xticklabels(conf_labels, rotation=30, ha='right')
ax.set_xlabel('final_confidence')
plt.tight_layout()
plt.show()

# --- Events by confidence bucket ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, conf_summary['n_events'], color=conf_colors)
ax.set_title('Redispatch events by confidence bucket')
ax.set_ylabel('Number of events')
for i, v in enumerate(conf_summary['n_events']):
    ax.annotate(f'{int(v):,}', (i, v), ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, conf_summary['n_events'].max() * 1.15)
ax.set_xticks(list(x))
ax.set_xticklabels(conf_labels, rotation=30, ha='right')
ax.set_xlabel('final_confidence')
plt.tight_layout()
plt.show()

### 14a. By geocoding source (`final_source`)

Which geocoder produced the coordinates for each matched plant, and how many events ride on each source. Same two views — distinct plants and total events.

In [ ]:
src_summary = (matched.assign(geocode_source=matched['geocode_source'].fillna('—'))
                      .groupby('geocode_source')
                      .agg(n_plants=('BETROFFENE_ANLAGE', 'nunique'),
                           n_events=('GRUND_DER_MASSNAHME', 'size'))
                      .sort_values('n_events', ascending=False))
src_summary['share_events'] = src_summary['n_events'] / src_summary['n_events'].sum()
src_summary['share_plants'] = src_summary['n_plants'] / src_summary['n_plants'].sum()
src_summary

In [ ]:
import matplotlib.cm as cm

src_colors = cm.tab10(np.linspace(0, 1, max(len(src_summary), 1)))
x = range(len(src_summary))

# --- Plants by source ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, src_summary['n_plants'], color=src_colors)
ax.set_title('Distinct matched plants by source')
ax.set_ylabel('Number of plants')
for i, v in enumerate(src_summary['n_plants']):
    ax.annotate(f'{int(v):,}', (i, v), ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, src_summary['n_plants'].max() * 1.15)
ax.set_xticks(list(x))
ax.set_xticklabels(src_summary.index, rotation=30, ha='right')
ax.set_xlabel('final_source (geocode_source)')
plt.tight_layout()
plt.show()

# --- Events by source ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, src_summary['n_events'], color=src_colors)
ax.set_title('Redispatch events by source')
ax.set_ylabel('Number of events')
for i, v in enumerate(src_summary['n_events']):
    ax.annotate(f'{int(v):,}', (i, v), ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, src_summary['n_events'].max() * 1.15)
ax.set_xticks(list(x))
ax.set_xticklabels(src_summary.index, rotation=30, ha='right')
ax.set_xlabel('final_source (geocode_source)')
plt.tight_layout()
plt.show()

### 14b. Source × confidence

Each geocoding source has its own confidence signature — e.g. exact MaStR/coordinate hits land in the top bucket, while fuzzy name lookups spread into the low-confidence tail. The crosstab below shows, per source, how its matched plants distribute across the `final_confidence` buckets; the stacked bar normalises each source to 100 % so the shapes are comparable regardless of how many plants each source contributed.

In [ ]:
# One row per distinct matched plant so each plant is counted once (not once per event).
plant_level = (matched.assign(geocode_source=matched['geocode_source'].fillna('—'))
                      .dropna(subset=['final_confidence'])
                      .drop_duplicates(subset=['BETROFFENE_ANLAGE']))

src_conf = pd.crosstab(plant_level['geocode_source'], plant_level['conf_bucket'])
src_conf = src_conf.reindex(columns=conf_labels, fill_value=0)
src_conf = src_conf.loc[src_conf.sum(axis=1).sort_values(ascending=False).index]
print('Plants per source × confidence bucket:')
display(src_conf)

# Normalise each source to 100 % to compare the confidence shape across sources.
src_conf_pct = src_conf.div(src_conf.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 5))
src_conf_pct.iloc[::-1].plot.barh(ax=ax, stacked=True, color=conf_colors, width=0.85)
ax.set_xlabel('Share of a source\'s matched plants')
ax.set_ylabel('')
ax.set_title('Confidence-bucket mix within each geocoding source')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8, title='final_confidence')
plt.tight_layout()
plt.show()

## 15. Geographic match rate **by ÜNB**

For each of the four German ÜNB, what share of the redispatch they were involved in could be tied to a geographic location via `plant_matcher_output`? Three yardsticks per ÜNB:

- **events** — share of their events that matched,
- **plants** — share of their distinct `BETROFFENE_ANLAGE` that matched,
- **MWh** — share of their `GESAMTE_ARBEIT_MWH` that matched.

A "geographic match" = the event joined a matcher row (`_merge == 'both'`) **and** that row carries coordinates (`lat` not null).

The two TSO roles in the data:
- **anfordernder** (`ANFORDERNDER_UENB`) — the ÜNB that *requested* the measure,
- **übertragender** (`ANWEISENDER_UENB`) — the ÜNB that *instructed / carried out* the measure on its grid.

`ANFORDERNDER_UENB` is messy (it contains neighbouring operators and `A & B` combinations), so membership is decided by **substring**: an event where the field is `"50Hertz & TenneT DE"` counts toward both 50Hertz and TenneT DE. Three figures below: (1) involved as **either** role, then (2) anfordernder only and (3) übertragender only.

In [ ]:
TSOS = ['50Hertz', 'Amprion', 'TenneT DE', 'TransnetBW']

# A geographic match = joined to a matcher row AND that row has coordinates.
is_geo = (joined['_merge'] == 'both') & joined['lat'].notna()


def uenb_match_table(membership_col=None, both=False):
    """Per-ÜNB matched-share of events, distinct plants and MWh volume.

    membership_col : column whose substring decides ÜNB membership
                     (used when both=False).
    both           : if True, an event counts for a ÜNB when its name appears
                     in EITHER ANFORDERNDER_UENB OR ANWEISENDER_UENB.
    """
    anf = joined['ANFORDERNDER_UENB'].fillna('')
    anw = joined['ANWEISENDER_UENB'].fillna('')
    rows = {}
    for tso in TSOS:
        if both:
            mask = anf.str.contains(tso, regex=False) | anw.str.contains(tso, regex=False)
        else:
            mask = joined[membership_col].fillna('').str.contains(tso, regex=False)
        sub = joined[mask]
        sub_geo = joined[mask & is_geo]

        n_ev, n_ev_m = len(sub), len(sub_geo)
        mwh = sub['GESAMTE_ARBEIT_MWH'].sum()
        mwh_m = sub_geo['GESAMTE_ARBEIT_MWH'].sum()
        n_pl = sub['BETROFFENE_ANLAGE'].nunique()
        n_pl_m = sub_geo['BETROFFENE_ANLAGE'].nunique()

        rows[tso] = {
            'events_%':  100 * n_ev_m / n_ev if n_ev else np.nan,
            'plants_%':  100 * n_pl_m / n_pl if n_pl else np.nan,
            'MWh_%':     100 * mwh_m / mwh if mwh else np.nan,
            'n_events':  n_ev,
            'n_plants':  n_pl,
            'MWh_total': mwh,
        }
    return pd.DataFrame(rows).T


def plot_uenb_match(tbl, title):
    metrics = ['events_%', 'plants_%', 'MWh_%']
    metric_colors = {'events_%': '#3a6ea5', 'plants_%': '#3aa856', 'MWh_%': '#cc4b3a'}
    x = np.arange(len(tbl))
    w = 0.26

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, m in enumerate(metrics):
        bars = ax.bar(x + (i - 1) * w, tbl[m], w,
                      label=m.replace('_%', ''), color=metric_colors[m])
        for b, v in zip(bars, tbl[m]):
            if pd.notna(v):
                ax.annotate(f'{v:.0f}%', (b.get_x() + b.get_width() / 2, v),
                            ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(tbl.index)
    ax.set_ylabel('Share matched to a location (%)')
    ax.set_ylim(0, 105)
    ax.set_title(title)
    ax.legend(title='matched share of', loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()


# (1) involved as EITHER role
tbl_either = uenb_match_table(both=True)
print('Involved as anfordernder OR übertragender:')
display(tbl_either.round(1))
plot_uenb_match(tbl_either, 'Geographic match rate by ÜNB — involved as either role')

# (2) anfordernder only
tbl_anf = uenb_match_table('ANFORDERNDER_UENB')
print('\nAs anfordernder (ANFORDERNDER_UENB):')
display(tbl_anf.round(1))
plot_uenb_match(tbl_anf, 'Geographic match rate by ÜNB — as anfordernder (requesting)')

# (3) übertragender only
tbl_anw = uenb_match_table('ANWEISENDER_UENB')
print('\nAs übertragender (ANWEISENDER_UENB):')
display(tbl_anw.round(1))
plot_uenb_match(tbl_anw, 'Geographic match rate by ÜNB — as übertragender (instructing)')

### 15b. Same, but only counting confident matches (`final_confidence > CONF_THRESHOLD`)

Identical to §15, except a "geographic match" now also requires the matcher's `final_confidence` to clear the **same threshold that feeds the high-confidence Excel** (`CONF_THRESHOLD` from §3a, currently **0.5**) — so the low-confidence, hand-wavy geocodes no longer count toward the matched share, and these bars line up exactly with what actually makes it into the export. Denominators (each ÜNB's total events / plants / MWh) are unchanged, so these bars sit at or below the §15 ones; the gap is exactly the volume that matched but only at low confidence.

(The cell reads `CONF_THRESHOLD` directly, so if you change the threshold in §3a this section follows automatically. The titles below print the live value.)

In [ ]:
# Use the SAME confidence threshold that feeds the high-confidence Excel (§3a).
# Falls back to 0.5 if §3a hasn't been run yet, so §15b never drifts from the export rule.
CONF_MIN = globals().get('CONF_THRESHOLD', 0.5)

# A geographic match now also requires a confident geocode.
is_geo_conf = (joined['_merge'] == 'both') & joined['lat'].notna() & (joined['final_confidence'] > CONF_MIN)


def uenb_match_table_conf(membership_col=None, both=False, geo_mask=is_geo_conf):
    """Same as uenb_match_table but with a swappable 'matched' mask."""
    anf = joined['ANFORDERNDER_UENB'].fillna('')
    anw = joined['ANWEISENDER_UENB'].fillna('')
    rows = {}
    for tso in TSOS:
        if both:
            mask = anf.str.contains(tso, regex=False) | anw.str.contains(tso, regex=False)
        else:
            mask = joined[membership_col].fillna('').str.contains(tso, regex=False)
        sub = joined[mask]
        sub_geo = joined[mask & geo_mask]

        n_ev, n_ev_m = len(sub), len(sub_geo)
        mwh = sub['GESAMTE_ARBEIT_MWH'].sum()
        mwh_m = sub_geo['GESAMTE_ARBEIT_MWH'].sum()
        n_pl = sub['BETROFFENE_ANLAGE'].nunique()
        n_pl_m = sub_geo['BETROFFENE_ANLAGE'].nunique()

        rows[tso] = {
            'events_%':  100 * n_ev_m / n_ev if n_ev else np.nan,
            'plants_%':  100 * n_pl_m / n_pl if n_pl else np.nan,
            'MWh_%':     100 * mwh_m / mwh if mwh else np.nan,
            'n_events':  n_ev,
            'n_plants':  n_pl,
            'MWh_total': mwh,
        }
    return pd.DataFrame(rows).T


# (1) involved as EITHER role
tbl_either_c = uenb_match_table_conf(both=True)
print(f'Involved as anfordernder OR übertragender (final_confidence > {CONF_MIN}):')
display(tbl_either_c.round(1))
plot_uenb_match(tbl_either_c, f'Match rate by ÜNB — either role (confidence > {CONF_MIN})')

# (2) anfordernder only
tbl_anf_c = uenb_match_table_conf('ANFORDERNDER_UENB')
print(f'\nAs anfordernder (final_confidence > {CONF_MIN}):')
display(tbl_anf_c.round(1))
plot_uenb_match(tbl_anf_c, f'Match rate by ÜNB — as anfordernder (confidence > {CONF_MIN})')

# (3) übertragender only
tbl_anw_c = uenb_match_table_conf('ANWEISENDER_UENB')
print(f'\nAs übertragender (final_confidence > {CONF_MIN}):')
display(tbl_anw_c.round(1))
plot_uenb_match(tbl_anw_c, f'Match rate by ÜNB — as übertragender (confidence > {CONF_MIN})')

## 16. Börse / Countertrade volume — the unmatchable market share

Two of the biggest "unmatched" buckets aren't data-quality misses at all — they are entries that *can never* carry a plant location:

- **Börse** — the wholesale market as `BETROFFENE_ANLAGE` (no physical site).
- **Countertrade** — `GRUND_DER_MASSNAHME` = `Strombedingter Countertrade DE-DKx / DE-NO2`, the cross-border market-based counterpart to redispatch.

The two overlap almost completely (countertrade is booked against `Börse`), so to avoid double counting we split them into **Countertrade** vs. **Börse (non-countertrade)**.

Charts below:
1. **What % of total redispatched volume (`GESAMTE_ARBEIT_MWH`) is lost to Börse/Countertrade, split by the instructing ÜNB (`ANWEISENDER_UENB`)** — shows which TSO this happens in most.
2. **Total volume by cause** as a pie + stacked bar, so the Börse/Countertrade slice sits in the context of the genuine grid-stress causes.

In [ ]:
# --- Classify each event's volume into Countertrade / Börse(market) / matchable ----
total_mwh = joined['GESAMTE_ARBEIT_MWH'].sum()

# 'b.rse' regex catches both the clean 'Börse' and the mojibake 'B�rse'.
borse_mask = joined['BETROFFENE_ANLAGE'].astype(str).str.contains(r'b.rse', case=False, regex=True, na=False)
ct_mask    = joined['GRUND_DER_MASSNAHME'].astype(str).str.contains('Countertrade', case=False, na=False)

market_class = np.where(ct_mask, 'Countertrade',
                np.where(borse_mask, 'Börse (Markt)', 'übrige (anlagenbezogen)'))
joined['_market_class'] = market_class

# --- Per-ÜNB volume lost to Börse/Countertrade, as % of the GRAND total volume ----
TSOS = ['50Hertz', 'Amprion', 'TenneT DE', 'TransnetBW']
mc = (joined[joined['ANWEISENDER_UENB'].isin(TSOS)]
      .pivot_table(index='ANWEISENDER_UENB', columns='_market_class',
                   values='GESAMTE_ARBEIT_MWH', aggfunc='sum', fill_value=0))
for col in ['Countertrade', 'Börse (Markt)']:
    if col not in mc.columns:
        mc[col] = 0.0
mc_share = 100 * mc[['Börse (Markt)', 'Countertrade']] / total_mwh   # % of grand total
mc_share = mc_share.loc[mc_share.sum(axis=1).sort_values(ascending=False).index]

combined_share = 100 * joined.loc[borse_mask | ct_mask, 'GESAMTE_ARBEIT_MWH'].sum() / total_mwh
print(f'Total redispatched volume                  : {total_mwh/1e6:6.2f} TWh')
print(f'Börse OR Countertrade share of total volume: {combined_share:5.1f} %')
display(mc_share.assign(**{'Σ % of total': mc_share.sum(axis=1)}).round(2))

fig, ax = plt.subplots(figsize=(9, 5))
seg_colors = {'Börse (Markt)': '#8a6dc0', 'Countertrade': '#cc4b3a'}
bottom = np.zeros(len(mc_share))
for seg in ['Börse (Markt)', 'Countertrade']:
    ax.bar(mc_share.index, mc_share[seg], bottom=bottom, color=seg_colors[seg], label=seg)
    bottom += mc_share[seg].values
for i, tot in enumerate(mc_share.sum(axis=1)):
    ax.annotate(f'{tot:.1f}%', (i, tot), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Share of TOTAL redispatched volume (%)')
ax.set_xlabel('Instructing ÜNB (ANWEISENDER_UENB)')
ax.set_title(f'Volume unmatchable as Börse / Countertrade by ÜNB\n'
             f'(together {combined_share:.1f}% of all {total_mwh/1e6:.1f} TWh)')
ax.legend(title='', loc='upper right')
plt.tight_layout()
plt.show()


# --- Total volume of redispatch by cause: pie + stacked bar -----------------------
vol_by_cause = (joined.groupby('GRUND_DER_MASSNAHME')['GESAMTE_ARBEIT_MWH']
                      .sum().sort_values(ascending=False) / 1e3)   # GWh

# Collapse the long tail of small causes into 'Sonstige' for a readable pie.
TOP_N = 6
top_causes = vol_by_cause.head(TOP_N)
rest = vol_by_cause.iloc[TOP_N:].sum()
pie_data = top_causes.copy()
if rest > 0:
    pie_data['Sonstige'] = rest

pie_colors = plt.cm.tab20(np.linspace(0, 1, len(pie_data)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6),
                               gridspec_kw={'width_ratios': [1.1, 1]})

wedges, _ = ax1.pie(pie_data.values, colors=pie_colors, startangle=90,
                    counterclock=False, wedgeprops=dict(width=0.45, edgecolor='white'))
ax1.set_title('Total redispatched volume by cause')
ax1.legend(wedges,
           [f'{c} — {v:,.0f} GWh ({v/vol_by_cause.sum():.1%})'
            for c, v in pie_data.items()],
           loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)

# Stacked single bar (100%-of-volume) next to it.
left = 0.0
for (cause, v), col in zip(pie_data.items(), pie_colors):
    ax2.barh(0, v, left=left, color=col, edgecolor='white')
    if v / pie_data.sum() > 0.04:
        ax2.text(left + v/2, 0, f'{v/vol_by_cause.sum():.0%}',
                 ha='center', va='center', fontsize=8,
                 color='white', fontweight='bold')
    left += v
ax2.set_xlim(0, pie_data.sum())
ax2.set_ylim(-0.5, 0.5)
ax2.set_yticks([])
ax2.set_xlabel('Redispatched volume (GWh)')
ax2.set_title('Volume share by cause (stacked)')
ax2.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

## 17. What gets dropped by the high-confidence filter

Same view as §12 (unmatched-by-category), but widened to **everything the high-confidence Excel/CSV excludes on the geocoding criterion** — i.e. every event that is *not* a confident plant match. That is the complement of the §3a keep-rule `_merge == 'both' & final_confidence > CONF_THRESHOLD`, so the dropped set is:

- **unmatched** events (no plant row at all) — split into the §12 categories `market / multi-plant bundle / reserve label / other`, **plus**
- **low-confidence match** — events that *did* join a plant row but at `final_confidence ≤ CONF_THRESHOLD`, so they're thrown out of the high-confidence export.

`CONF_THRESHOLD` is read straight from §3a (currently 0.5), so this chart always tracks whatever threshold feeds the high-confidence Excel.

In [ ]:
# Threshold that feeds the high-confidence Excel (§3a). Fallback to 0.5 if §3a not run.
THRESH = globals().get('CONF_THRESHOLD', 0.5)

# Dropped on the geocoding criterion = NOT (matched AND confident).
dropped = joined[~((joined['_merge'] == 'both') & (joined['final_confidence'] > THRESH))].copy()

# Classify: matched-but-low-confidence -> own bucket; truly unmatched -> §12 categories.
is_matched = dropped['_merge'] == 'both'
dropped['drop_class'] = np.where(
    is_matched,
    'low-confidence match',
    dropped['BETROFFENE_ANLAGE'].map(classify_unmatched),   # reuse §12 classifier
)

drop_dist = (dropped.groupby('drop_class')
                    .agg(n_events=('GRUND_DER_MASSNAHME', 'size'),
                         energy_gwh=('GESAMTE_ARBEIT_MWH', lambda s: s.sum()/1e3),
                         n_distinct_names=('BETROFFENE_ANLAGE', 'nunique'))
                    .sort_values('n_events', ascending=False))
drop_dist['share_of_dropped'] = drop_dist['n_events'] / drop_dist['n_events'].sum()
drop_dist['share_of_all_events'] = drop_dist['n_events'] / len(joined)

print(f'CONF_THRESHOLD = {THRESH}')
print(f'events dropped by the high-confidence filter: {len(dropped):,} '
      f'({len(dropped)/len(joined):.1%} of all events)')
display(drop_dist.round(3))

cls_colors = {'market': '#8a6dc0', 'multi-plant bundle': '#cc4b3a',
              'reserve label': '#e09d3c', 'other': '#3a6ea5',
              'low-confidence match': '#46a5b3'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

dd = drop_dist.copy()
axes[0].bar(dd.index, dd['n_events'], color=[cls_colors.get(c, '#888') for c in dd.index])
axes[0].set_ylabel('Number of dropped events')
axes[0].set_title(f"Events excluded from high-confidence export by category\n"
                  f"(total {len(dropped):,}, conf ≤ {THRESH} or unmatched)")
for i, (n, s) in enumerate(zip(dd['n_events'], dd['share_of_dropped'])):
    axes[0].annotate(f'{int(n):,}\n{s:.0%}', (i, n), ha='center', va='bottom', fontsize=8)
axes[0].set_ylim(0, dd['n_events'].max() * 1.18)
plt.setp(axes[0].get_xticklabels(), rotation=20, ha='right')

axes[1].bar(dd.index, dd['energy_gwh'], color=[cls_colors.get(c, '#888') for c in dd.index])
axes[1].set_ylabel('Total energy (GWh)')
axes[1].set_title('Dropped energy by category')
for i, e in enumerate(dd['energy_gwh']):
    axes[1].annotate(f'{e:,.0f}', (i, e), ha='center', va='bottom', fontsize=8)
axes[1].set_ylim(0, dd['energy_gwh'].max() * 1.15)
plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.show()

## 18. Whole-fleet event profile — size, duration, seasonality

Everything here is on the **full** redispatch table (`joined`, matched + unmatched alike — no plant filter), so it describes the redispatch *activity* itself rather than the geolocated subset.

- **18a Size** — how big is a typical event, by mean power (`MITTLERE_LEISTUNG_MW`) and by total work (`GESAMTE_ARBEIT_MWH`). Both are extremely heavy-tailed (a handful of huge conventional interventions, a long tail of tiny ones), so the histograms use a log x-axis and the table reports percentiles, not just the mean.
- **18b Duration** — `duration_h` (`ts_end − ts_start`), again log-scaled.
- **18c Seasonality** — collapsing every year onto a single Jan–Dec axis: what **share of events** and what **share of volume (MWh)** falls in each calendar month. This is the "when in the year does redispatch happen" view (winter wind / cold-spell congestion vs. calmer summer).

In [ ]:
# ===== 18a + 18b: size & duration distributions over ALL events =====================
power = pd.to_numeric(joined['MITTLERE_LEISTUNG_MW'], errors='coerce')
work  = pd.to_numeric(joined['GESAMTE_ARBEIT_MWH'], errors='coerce')
dur   = pd.to_numeric(joined['duration_h'], errors='coerce')

# Summary table (percentiles tell the heavy-tail story better than the mean).
size_summary = pd.DataFrame({
    'mean power (MW)':  power,
    'total work (MWh)': work,
    'duration (h)':     dur,
}).describe(percentiles=[.05, .25, .5, .75, .9, .95, .99]).round(1)
print(f'events total: {len(joined):,}  '
      f'(power non-null {power.notna().sum():,}, '
      f'work {work.notna().sum():,}, duration {dur.notna().sum():,})')
display(size_summary)

# Only positive, finite values can go on a log axis.
power_pos = power[(power > 0) & np.isfinite(power)]
work_pos  = work[(work > 0) & np.isfinite(work)]
dur_pos   = dur[(dur > 0) & np.isfinite(dur)]

def logbins(s, n=40):
    return np.logspace(np.log10(s.min()), np.log10(s.max()), n)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, s, title, xlab, col in [
    (axes[0], power_pos, 'Mean power per event',  'MITTLERE_LEISTUNG_MW (log)', '#3a6ea5'),
    (axes[1], work_pos,  'Total work per event',  'GESAMTE_ARBEIT_MWH (log)',   '#5a4d80'),
    (axes[2], dur_pos,   'Duration per event',    'duration_h (log)',           '#cc4b3a'),
]:
    ax.hist(s, bins=logbins(s), color=col, alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel(xlab)
    ax.set_ylabel('Number of events')
    ax.axvline(s.median(), color='k', ls='--', lw=1)
    ax.annotate(f'median {s.median():,.1f}', (s.median(), ax.get_ylim()[1]*0.9),
                fontsize=8, rotation=90, va='top', ha='right')

fig.suptitle('18a–b  Size & duration of all redispatch events (2013–2025, log x-axis)',
             y=1.03, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 18c: intra-year seasonality — which calendar months carry the redispatch =====
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

season = joined.dropna(subset=['ts_start']).copy()
season['month_no'] = season['ts_start'].dt.month

month_stat = (season.groupby('month_no')
                    .agg(n_events=('GRUND_DER_MASSNAHME', 'size'),
                         volume_gwh=('GESAMTE_ARBEIT_MWH', lambda s: s.sum()/1e3))
                    .reindex(range(1, 13), fill_value=0))
month_stat.index = MONTHS
month_stat['share_events'] = month_stat['n_events'] / month_stat['n_events'].sum()
month_stat['share_volume'] = month_stat['volume_gwh'] / month_stat['volume_gwh'].sum()

# Number of distinct years each calendar month is observed in, so a part-year at the
# end of the series (e.g. data stopping mid-year) doesn't read as a fake seasonal dip.
month_stat['n_years_observed'] = season.groupby('month_no')['ts_start'].apply(
    lambda s: s.dt.year.nunique()).reindex(range(1, 13)).values

print('Redispatch by calendar month (all years pooled):')
display(month_stat.assign(
    share_events=lambda d: (d['share_events']*100).round(1),
    share_volume=lambda d: (d['share_volume']*100).round(1)).round(1))

# --- Grouped bars: share of events vs share of volume, per month ---
x = np.arange(12)
w = 0.4
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, month_stat['share_events']*100, w, label='share of events',
       color='#3a6ea5')
ax.bar(x + w/2, month_stat['share_volume']*100, w, label='share of volume (MWh)',
       color='#cc4b3a')
ax.axhline(100/12, color='k', ls=':', lw=1)
ax.annotate('flat (8.3% / month)', (11.4, 100/12), fontsize=8, va='bottom', ha='right')
ax.set_xticks(x)
ax.set_xticklabels(MONTHS)
ax.set_ylabel('Share of yearly total (%)')
ax.set_title('18c  Seasonality of redispatch — share by calendar month (2013–2025 pooled)')
for i in x:
    ax.annotate(f"{month_stat['share_events'].iloc[i]*100:.0f}",
                (i - w/2, month_stat['share_events'].iloc[i]*100),
                ha='center', va='bottom', fontsize=7)
    ax.annotate(f"{month_stat['share_volume'].iloc[i]*100:.0f}",
                (i + w/2, month_stat['share_volume'].iloc[i]*100),
                ha='center', va='bottom', fontsize=7)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# --- Seasonal radial view of volume share, to read winter vs summer at a glance ---
fig = plt.figure(figsize=(7, 7))
axp = fig.add_subplot(111, projection='polar')
theta = np.linspace(0, 2*np.pi, 12, endpoint=False) + np.pi/2   # Jan at top
theta = (np.pi/2 - np.arange(12) * 2*np.pi/12)                   # clockwise from top
vals = month_stat['share_volume'].values * 100
bars = axp.bar(theta, vals, width=2*np.pi/12*0.9,
               color=plt.cm.coolwarm(vals / vals.max()), edgecolor='white')
axp.set_xticks(theta)
axp.set_xticklabels(MONTHS)
axp.set_yticklabels([])
axp.set_title('Share of redispatched volume by month (MWh)', va='bottom')
plt.tight_layout()
plt.show()